# 01 — Dataset Verification

Audit the Paderborn raw-data folder structure and expected MATLAB recording counts before processing. This notebook verifies bearing folders, measurement files and the expected dataset inventory.

Place the original Paderborn data under `data/raw/`; the raw recordings are intentionally not included in Git.

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# STAGE 1: DATASET VERIFICATION
# ============================================================

RAW_DIR = Path("../data/raw")

healthy_bearings = [
    "K001", "K002", "K003", "K004", "K005", "K006"
]

outer_bearings = [
    "KA01", "KA03", "KA04", "KA05", "KA06",
    "KA07", "KA08", "KA09", "KA15", "KA16",
    "KA22", "KA30"
]

inner_bearings = [
    "KI01", "KI03", "KI04", "KI05", "KI07",
    "KI08", "KI14", "KI16", "KI17", "KI18", "KI21"
]

both_bearings = [
    "KB23", "KB24", "KB27"
]

expected_bearings = (
    healthy_bearings
    + outer_bearings
    + inner_bearings
    + both_bearings
)

print("Expected bearings:", len(expected_bearings))

existing_bearings = sorted([
    p.name
    for p in RAW_DIR.iterdir()
    if p.is_dir()
])

print("Found bearings:", len(existing_bearings))

missing = sorted(
    set(expected_bearings) - set(existing_bearings)
)

unexpected = sorted(
    set(existing_bearings) - set(expected_bearings)
)

print("\nMissing folders:")
print(missing if missing else "None")

print("\nUnexpected folders:")
print(unexpected if unexpected else "None")

In [ ]:
# ============================================================
# COUNT .MAT FILES
# ============================================================

records = []

for bearing in expected_bearings:

    bearing_dir = RAW_DIR / bearing

    if bearing in healthy_bearings:
        category = "healthy"
    elif bearing in outer_bearings:
        category = "outer"
    elif bearing in inner_bearings:
        category = "inner"
    else:
        category = "both"

    if not bearing_dir.exists():
        records.append({
            "bearing": bearing,
            "category": category,
            "mat_files": 0,
            "status": "MISSING FOLDER"
        })
        continue

    mat_files = list(bearing_dir.glob("*.mat"))

    records.append({
        "bearing": bearing,
        "category": category,
        "mat_files": len(mat_files),
        "status": "OK" if len(mat_files) == 80 else "CHECK"
    })

verification_df = pd.DataFrame(records)

display(verification_df)

In [ ]:
# ============================================================
# VERIFICATION SUMMARY
# ============================================================

print("Total bearings:", len(verification_df))
print("Total .mat files:", verification_df["mat_files"].sum())

print("\nBearings by category:")
print(
    verification_df["category"]
    .value_counts()
)

print("\nFile count distribution:")
print(
    verification_df["mat_files"]
    .value_counts()
    .sort_index()
)

print("\nBearings requiring attention:")
display(
    verification_df[
        verification_df["status"] != "OK"
    ]
)